# **ETL SILVER**

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW product_cleaning AS
select
  nombre_mercado,
  nombre_producto,
  unidad_medida,
  case
    when precio_inicio rlike '^-?[0-9]+\\.?[0-9]*$' and precio_inicio::double between 0 and 2147483648 then precio_inicio::double
    else null
  end as precio_inicio_clean,
  case
    when precio_fin rlike '^-?[0-9]+\\.?[0-9]*$' and precio_fin::double between 0 and 2147483648 then precio_fin::double
    else null
  end as precio_fin_clean,
  case
    when inicio_fecha_precios RLIKE '^[^a-zA-Z]+$' then to_date(inicio_fecha_precios, 'dd-MM-yyyy')
    else null
  end as inicio_fecha_precios_clean,
  case
    when fin_fecha_precios RLIKE '^[^a-zA-Z]+$' then to_date(fin_fecha_precios, 'dd-MM-yyyy')
    else null
  end as fin_fecha_precios_clean
from
  products.bronze.product_list_markets_bronze
where
  error = "null"
  or nombre_mercado != 'null'
  or nombre_producto != 'null'
  or unidad_medida != 'null'
  or precio_inicio != 'null'
  or precio_fin != 'null'
  or inicio_fecha_precios != 'null'
  or fin_fecha_precios != 'null'

In [0]:
%sql
select * from product_cleaning


In [0]:
%sql
create or replace temporary view product_clean_values as(

    with clean_percentile as(
        select 
            nombre_mercado,
            nombre_producto,
            unidad_medida,
            percentile_approx(precio_inicio_clean, 0.01) as precio_inicio_clean_001,
            percentile_approx(precio_inicio_clean, 0.99) as precio_inicio_clean_099,
            percentile_approx(precio_fin_clean, 0.01) as precio_fin_clean_001,
            percentile_approx(precio_fin_clean, 0.99) as precio_fin_clean_099,
            inicio_fecha_precios_clean,
            fin_fecha_precios_clean
        from product_cleaning
        where precio_inicio_clean > 0 and precio_fin_clean > 0
        group by
            nombre_mercado,
            nombre_producto,
            unidad_medida,
            inicio_fecha_precios_clean,
            fin_fecha_precios_clean

    )

    select 
    pc.nombre_mercado,
    pc.nombre_producto,
    pc.unidad_medida,
    pc.precio_inicio_clean,
    pc.precio_fin_clean,
    pc.inicio_fecha_precios_clean,
    pc.fin_fecha_precios_clean
    from product_cleaning pc
    inner join clean_percentile cpe
        on pc.nombre_mercado = cpe.nombre_mercado 
        and pc.nombre_producto = cpe.nombre_producto 
        and pc.unidad_medida = cpe.unidad_medida 
        and pc.inicio_fecha_precios_clean = cpe.inicio_fecha_precios_clean 
    where 
        pc.precio_inicio_clean between cpe.precio_inicio_clean_001 and cpe.precio_inicio_clean_099
        and pc.precio_fin_clean between cpe.precio_fin_clean_001 and cpe.precio_fin_clean_099

)

In [0]:
%sql
select * from product_clean_values

In [0]:
%sql
create or replace temporary view bronze_products_EDA as
(
  with datos_deduplicados AS (
    SELECT
      *,
      ROW_NUMBER() OVER (
          PARTITION BY
            nombre_mercado,
            nombre_producto,
            unidad_medida,
            inicio_fecha_precios_clean,
            fin_fecha_precios_clean
          ORDER BY nombre_mercado asc
        ) as rn
    FROM
      product_clean_values
  )

  
  select
    lower(trim(nombre_mercado)) as nombre_mercado,
    lower(trim(nombre_producto)) as nombre_producto,
    lower(trim(unidad_medida)) as unidad_medida,
    "LPS" as currency,
    COALESCE(CAST(precio_inicio_clean AS DECIMAL(15, 2)), 0) as precio_inicio,
    COALESCE(CAST(precio_fin_clean AS DECIMAL(15, 2)), 0) as precio_fin,
    inicio_fecha_precios_clean as inicio_fecha_precios,
    fin_fecha_precios_clean as fin_fecha_precios
  from
    datos_deduplicados
  where
    precio_inicio_clean > 0
    and precio_fin_clean > 0
    and unidad_medida != 'null'
    and inicio_fecha_precios_clean is not null
    and fin_fecha_precios_clean is not null
    and rn = 1
  order by inicio_fecha_precios asc
)

In [0]:
%sql
select * from bronze_products_EDA


In [0]:
%sql
create or replace temporary view bronze_products_silver as
(
  select
    nombre_mercado,
    case
      when nombre_mercado like '%comayagüela%' then 'comayagüela'
      when nombre_mercado like '%san isidro%' then 'comayagüela'
      when nombre_mercado like '%las americas%' then 'comayagüela'
      when nombre_mercado like '%ahorro ferias del pueblo%' then 'tegucigalpa'
      when nombre_mercado like '%feria del agricultor%' then 'tegucigalpa'
      when nombre_mercado like '%tegucigalpa%' then 'tegucigalpa'
      when nombre_mercado like '%san pedro%' then 'san pedro'
      when nombre_mercado like '%feria agropecuaria de villanueva%' then 'villanueva'
    end as city,
    case
      when city like 'comayagüela' then 'francisco morazan'
      when city like 'tegucigalpa' then 'francisco morazan'
      when city like 'san pedro' then 'cortes'
      when city like 'villanueva' then 'cortes'
    end as department,
    nombre_producto,
    unidad_medida,
    currency,
    precio_inicio,
    precio_fin,
    inicio_fecha_precios,
    fin_fecha_precios
  from
    bronze_products_EDA
)

In [0]:
%sql
select * from bronze_products_silver

In [0]:
%sql
delete from products.silver.product_list_silver

In [0]:
%sql

merge into products.silver.product_list_silver as target
using (
  select
    nombre_producto,
    unidad_medida,
    currency,
    precio_inicio,
    precio_fin,
    nombre_mercado,
    city,
    department,
    inicio_fecha_precios,
    fin_fecha_precios
  from
    bronze_products_silver)
  as source
on target.product_name = source.nombre_producto
and target.product_unit = source.unidad_medida
and target.start_date_price = source.precio_inicio
and target.market_name = source.nombre_mercado
and target.city = source.city
and target.department = source.department
and target.start_date_market = source.inicio_fecha_precios
when matched then
update set 
  target.product_name = source.nombre_producto,
  target.product_unit = source.unidad_medida,
  target.price_currency = source.currency,
  target.start_date_price = source.precio_inicio,
  target.end_date_price = source.precio_fin,
  target.market_name = source.nombre_mercado,
  target.city = source.city,
  target.department = source.department,
  target.start_date_market = source.inicio_fecha_precios,
  target.end_date_market = source.fin_fecha_precios
when not matched then 
insert (
  product_name,
  product_unit,
  price_currency,
  start_date_price,
  end_date_price,
  market_name,
  city,
  department,
  start_date_market,
  end_date_market)
  values (
  source.nombre_producto,
  source.unidad_medida,
  source.currency,
  source.precio_inicio,
  source.precio_fin,
  source.nombre_mercado,
  source.city,
  source.department,
  source.inicio_fecha_precios,
  source.fin_fecha_precios)


In [0]:
%sql
OPTIMIZE products.silver.product_list_silver;

In [0]:
%sql
select * from products.silver.product_list_silver